<a href="https://colab.research.google.com/github/asamsaics24-crypto/BIS-LAB-/blob/main/grey_wolf_optimizer_application_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
np.random.seed(42)
X = np.random.randint(1, 11, (200, 4))


y = ((X[:,0] + X[:,1] + X[:,2] + X[:,3]) >= 22).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def fitness(pos):
    mask = pos > 0.5
    if not mask.any():
        return 1
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train[:,mask], y_train)
    pred = model.predict(X_train[:,mask])
    return 1 - accuracy_score(y_train, pred)

def GWO(n=10, iterations=20, dim=4):
    wolves = np.random.rand(n, dim)
    alpha = np.zeros(dim)
    beta = np.zeros(dim)
    delta = np.zeros(dim)
    scores = [float('inf')]*3

    for t in range(iterations):
        for w in wolves:
            f = fitness(w)
            if f < scores[0]:
                scores = [f, scores[0], scores[1]]
                alpha, beta, delta = w.copy(), alpha.copy(), beta.copy()
            elif f < scores[1]:
                scores[1], beta = f, w.copy()
            elif f < scores[2]:
                scores[2], delta = f, w.copy()

        a = 2 - 2*t/iterations
        for i in range(n):
            new = []
            for leader in [alpha, beta, delta]:
                A = 2*a*np.random.rand(dim)-a
                C = 2*np.random.rand(dim)
                new.append(leader-A*abs(C*leader-wolves[i]))
            wolves[i] = np.clip(np.mean(new, axis=0), 0, 1)

    return alpha

best = GWO()
selected = best > 0.5
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train[:,selected], y_train)
pred = model.predict(X_test[:,selected])
print("Student Performance Prediction using GWO")
print("Algorithm: Grey Wolf Optimizer + Decision Tree")
print("Total Students:", len(X))
print("Selected Features:", selected.sum())
print("Classification Accuracy:", round(accuracy_score(y_test,pred)*100,2), "%")
print("High Performance Students:", np.sum(pred == 1))
print("Low Performance Students:", np.sum(pred == 0))


Student Performance Prediction using GWO
Algorithm: Grey Wolf Optimizer + Decision Tree
Total Students: 200
Selected Features: 4
Classification Accuracy: 80.0 %
High Performance Students: 19
Low Performance Students: 21
